# Portfolio Project 2, Part 2: Your Chatbot on Telegram or Streamlit
### AITCD001 Chatbots Development · Learning Outcome 3 · United African Institute of Technology

**Due Monday 16 November 2026, 23:59; closes Thursday 19 November 2026, 23:59.** Marked out of 10 with the curriculum's integrated rubric, rows
3.2, *Integrate NLP into chatbot logic*, shown working on Telegram or Streamlit.

Cells marked **YOUR DOMAIN** are replaced with your own chatbot, classifier and threshold. Cells
marked **PROVIDED** connect them and put them on Telegram: read them, do not change them.

The worked example is a **fictional** bus ticket desk. It may not be submitted.

**Telegram is preferred.** If Telegram does not work for you (for example, @BotFather refuses a new bot), put the same chatbot on a Streamlit web page: run Steps 1 to 3 here to test it, then follow `How to deploy.md` in the folder *Portfolio Project 2 Part 2 - Streamlit Starter*. Same rubric, same marks.

**Steps:** File, then Save a copy in Drive. Paste your chatbot (Step 1), upload `intent_classifier.joblib`
and set the threshold (Step 2), choose your `TRY` messages (Step 3), run everything from the top
(Runtime, then Run all), then start the bot (Step 5).

**Never** type the bot token into a cell: it goes in Colab's secrets only.

---

---

## Step 1: Your chatbot

**YOUR DOMAIN.** Paste the chatbot of Portfolio Project 1. The worked example is the fictional City Bus Desk; it may not be submitted.

In [6]:
# AITCD001, Week 7: a rule-based chatbot for a domain of my own
import re
from datetime import datetime

print("ready")

# STEP 1. YOUR DOMAIN: replace these three lines with your own.
BOT_NAME = "ULK Exam & Registry Desk"
USERS = "undergraduate students at the Independent University of Kigali (ULK)"
TOPICS = "exam timetables, clearance fees, transcript collection and booking a registry appointment"

print(f"{BOT_NAME}: a chatbot for {USERS}.")
print(f"It can help with {TOPICS}.")

# STEP 2. YOUR DOMAIN: your intents, in priority order. At least five.
RULES = [
    ("cancel",     r"\bcancel\w*\b|\bcall it off\b|\bstop\b"),
    ("book",       r"\bbook\w*\b|\bappointments?\b|\breserv\w*\b|\bregistry\b|\bslot\b"),
    ("timetable",  r"\btimetables?\b|\bexam\w*\b|\bschedules?\\b|\bwhat ?time\b|\bibizamini\b"),
    ("fees",       r"\bfees?\b|\bclearance\b|\bprices?\\b|\bcosts?\b|\bhow ?much\b|\bangahe\b"),
    ("transcript", r"\btranscripts?\b|\bresults?\b|\bgrades?\b|\bmarks?\b|\bamanota\b"),
]

print(len(RULES), "intents in the rule table")

# STEP 3. PROVIDED: cleaning, matching, and several requests in one message (Weeks 3 to 5).
def normalise(text):
    """Remove punctuation (keeping the colon of 21:00), lowercase, trim."""
    return re.sub(r"[^\w\s:]", "", text).lower().strip()


def match(message):
    """The tag of the first rule that matches, or "fallback"."""
    text = normalise(message)
    for tag, pattern in RULES:
        if re.search(pattern, text):
            return tag
    return "fallback"


CONNECTORS = r"\band\b|\bbut\b|\balso\b|\bthen\b|[?]"


def split_requests(message):
    """Break one message into the separate requests it contains."""
    requests = []
    for part in re.split(CONNECTORS, message.lower()):
        part = normalise(part)
        if part != "":
            requests.append(part)
    return requests


def route(message):
    """One (intent, request) pair for each request understood."""
    found = []
    seen = []
    for request in split_requests(message):
        tag = match(request)
        if tag != "fallback" and tag not in seen:
            seen.append(tag)
            found.append((tag, request))
    return found


print(route("when are exams and how much is clearance"))

# STEP 4. YOUR DOMAIN: one handler per intent, then the router.
FEES = {
    "clearance": "5,000 RWF",
    "transcript": "10,000 RWF",
    "re-sit": "15,000 RWF",
    "resit": "15,000 RWF",
    "id": "3,000 RWF",
}
FEE_ITEMS = r"\b(" + "|".join(FEES) + r")\b"

def handle_timetable(text):
    return "Main exams run Monday to Friday in two sessions: Morning (08:30–11:30) and Afternoon (14:00–17:00)."

def handle_fees(text):
    """Reads the message: returns the exact fee if a specific item is named."""
    found = re.search(FEE_ITEMS, text)
    if found:
        item = found.group(1)
        return f"The fee for {item} is {FEES[item]}, payable via Irembo or BK."
    return "Clearance is 5,000 RWF, Transcript is 10,000 RWF, Exam Re-sit is 15,000 RWF, and Student ID replacement is 3,000 RWF."

def handle_transcript(text):
    return "Provisional transcripts are issued at the Academic Registry (Block B, Room 104) within 3 working days after clearance."

def handle_book(text):
    """Starts the registry appointment conversation (Step 5)."""
    if flow["step"] is not None:
        return "Your appointment booking is already under way."
    return start_flow()

def handle_cancel(text):
    if flow["step"] is not None:
        reset_flow()
        return "Your appointment booking has been cancelled. Nothing was booked."
    return "There is no appointment booking in progress to cancel."

HANDLERS = {
    "cancel":     handle_cancel,
    "timetable":  handle_timetable,
    "fees":       handle_fees,
    "transcript": handle_transcript,
    "book":       handle_book,
}

ASKS = ["book"]  # intents whose reply is a question: it goes last

for tag, pattern in RULES:
    if tag not in HANDLERS:
        print("No handler for:", tag)
print(len(HANDLERS), "intents routed")

# STEP 5a. YOUR DOMAIN: the questions your ONE conversation asks, in order.
SERVICES = r"\b(clearance|transcript|re-sit|resit|id)\b"

FLOW_STEPS = [
    ("service", "Which service do you need? Clearance, transcript, resit, or ID?", SERVICES),
    ("day",     "Which day? Today or tomorrow?", r"\b(today|tomorrow)\b"),
    ("session", "Morning or afternoon?", r"\b(morning|afternoon)\b"),
]

DONE = "Appointment booked: "
NOT_DONE = "No problem. No appointment has been booked."

print(len(FLOW_STEPS), "questions, then a confirmation")

# STEP 5b. PROVIDED: the flow (Weeks 4 and 5), now driven by FLOW_STEPS,
# and ending with a real confirmation: nothing is done until the user says yes.
flow = {"step": None, "collected": {}}
YES = r"\b(yes|yeah|yego|correct|ok|okay)\b"     # yego: yes
NO = r"\b(no|nope|oya|wrong)\b"                  # oya: no


def start_flow():
    flow["step"] = 0
    flow["collected"] = {}
    return FLOW_STEPS[0][1]


def reset_flow():
    flow["step"] = None
    flow["collected"] = {}


def summary():
    return ", ".join(flow["collected"].values())


def continue_flow(message):
    """One answer: store it, ask the next question, and at the end, confirm."""
    text = normalise(message)
    if flow["step"] == "confirm":
        if re.search(YES, text):
            booked = summary()
            reset_flow()
            return DONE + booked + ". Thank you."
        if re.search(NO, text):
            reset_flow()
            return NOT_DONE
        return "Please answer yes or no."
    name, question, pattern = FLOW_STEPS[flow["step"]]
    found = re.search(pattern, text)
    if not found:
        return question                          # not a valid answer: ask again
    flow["collected"][name] = found.group()
    flow["step"] = flow["step"] + 1
    if flow["step"] < len(FLOW_STEPS):
        return FLOW_STEPS[flow["step"]][1]       # the next question
    flow["step"] = "confirm"
    return "So: " + summary() + ". Shall I book it? Yes or no."


print("Bot:", start_flow())
for answer in ["transcript please", "tomorrow", "morning", "yes"]:
    print("You:", answer)
    print("Bot:", continue_flow(answer))

# STEP 6. PROVIDED, except FOLLOW_UPS: memory, a name, a greeting, follow ups (Week 5).
# Run this cell ONCE: it adds the name and greeting rules to RULES.
context = {"name": None, "last_intent": None, "fallbacks": 0}

def reset_context():
    context["name"] = None
    context["last_intent"] = None
    context["fallbacks"] = 0

NAME = r"\bmy name is (\w+)"

def handle_name(text):
    found = re.search(NAME, text)
    if found:
        context["name"] = found.group(1).capitalize()
        return f"Nice to meet you, {context['name']}."
    return "Sorry, I did not catch your name."

def handle_greeting(text):
    if context["name"] is not None:
        return f"Muraho, {context['name']}! How can I help?"
    return (f"Muraho! I am the {BOT_NAME} assistant, a computer program. "
            f"I can help with {TOPICS}.")

RULES.insert(0, ("name", r"\bmy name is\b"))
RULES.append(("greeting", r"\bhello\b|\bhi\b|\bhey\b|\bmuraho\b|\bmwaramutse\b|\bmwiriwe\b"))
HANDLERS["name"] = handle_name
HANDLERS["greeting"] = handle_greeting

# YOUR DOMAIN: an intent, and the pattern that continues it ("and transcript?" after a fee).
FOLLOW_UPS = {"fees": FEE_ITEMS}

def follow_up(text):
    """A message with no intent of its own, read in the light of the last one."""
    last = context["last_intent"]
    if last in FOLLOW_UPS and re.search(FOLLOW_UPS[last], text):
        return [(last, text)]
    return []

print(len(RULES), "rules,", len(HANDLERS), "intents routed")

# STEP 7. PROVIDED: validation, errors, the fallback and a person (Week 6).
MAX_LENGTH = 300
ASK_AGAIN = f"Please type a question. I can help with {TOPICS}."
IN_WORDS = "Please type your question in words."
TOO_LONG = "That is a long message. Could you ask one thing at a time?"


def validate(message):
    """Return a reply if the message cannot be used; None if it can."""
    if message is None:
        return ASK_AGAIN
    text = message.strip()
    if text == "":
        return ASK_AGAIN
    if normalise(text) == "":
        return IN_WORDS
    if len(text) > MAX_LENGTH:
        return TOO_LONG
    return None


ERROR_LOG = []
SORRY = "Sorry, something went wrong on my side. Please try again, or type 'human'."


def log_error(message, error):
    ERROR_LOG.append({"time": datetime.now().strftime("%H:%M:%S"), "message": message,
                      "error": type(error).__name__ + ": " + str(error)})


def safe_handle(tag, request):
    """Run one handler. If it raises an error, log it and apologise instead of crashing."""
    try:
        return HANDLERS[tag](request)
    except Exception as error:
        log_error(request, error)
        return SORRY


def respond(found):
    """Answer every request found; a reply that asks a question goes last."""
    answers = []
    questions = []
    for tag, request in found:
        reply = safe_handle(tag, request)
        if tag in ASKS:
            questions.append(reply)
        else:
            answers.append(reply)
    return " ".join(answers + questions)


UNMATCHED = []
FALLBACK = f"Sorry, I did not understand. I can help with {TOPICS}."
STILL_LOST = f"Sorry, I still did not understand. Type 'human' for a member of staff, or ask about {TOPICS}."


def fallback(message):
    UNMATCHED.append(message)
    context["fallbacks"] = context["fallbacks"] + 1
    if context["fallbacks"] >= 2:
        return STILL_LOST
    return FALLBACK


EXIT_WORDS = ("bye", "goodbye", "exit", "quit", "thanks", "murakoze")
HUMAN_WORDS = ("human", "staff", "agent", "person")
HANDOVER = "Connecting you to a member of staff. They can see this conversation."

print("validation, error handling and the fallback are ready")

# STEP 7, continued. PROVIDED: one complete turn, in the Week 6 order.
def reply_to(message):
    problem = validate(message)                  # 0. unusable input
    if problem is not None:
        return problem
    text = normalise(message)

    if text in EXIT_WORDS:                       # 1. the exit, at any point
        name = context["name"]
        reset_flow()
        reset_context()
        if name is not None:
            return f"Thank you, {name}. Murakoze."
        return "Thank you for contacting us. Murakoze."

    if text in HUMAN_WORDS:                      # 2. a person, at any point
        reset_flow()
        return HANDOVER

    if flow["step"] is not None:                 # 3. the flow sees it first
        step_before = flow["step"]
        flow_reply = continue_flow(message)
        if flow["step"] != step_before:
            return flow_reply
        found = route(message)
        if len(found) == 0:
            return flow_reply
        reply = respond(found)
        if flow["step"] is not None:
            reply = reply + " " + flow_reply
        return reply

    found = route(message)                       # 4. route
    if len(found) == 0:
        found = follow_up(text)                  # 5. a follow up
    if len(found) == 0:
        return fallback(message)                 # 6. the fallback, counted

    context["fallbacks"] = 0
    last_tag, last_request = found[-1]
    context["last_intent"] = last_tag
    return respond(found)


def run_conversation(turns):
    """Play a conversation from a clean start, printing both sides."""
    reset_flow()
    reset_context()
    for message in turns:
        print("You:", message)
        print("Bot:", reply_to(message))
    print()


run_conversation(["muraho", "how much is clearance", "and transcript?", "bye"])

# STEP 8. YOUR DOMAIN: ten test conversations. Write what you EXPECT on paper first.
TESTS = [
    ('A greeting, a name, a goodbye',
     ['muraho', 'my name is aline', 'bye']),
    ('Two requests in one message',
     ['when are exams and how much is clearance']),
    ('An appointment booking, confirmed',
     ['i would like to book an appointment', 'transcript', 'tomorrow', 'morning', 'yes']),
    ('An appointment booking, refused at the end',
     ['book registry slot', 'clearance', 'today', 'afternoon', 'no']),
    ('A side question during the booking',
     ['book an appointment', 'resit', 'how much is resit', 'today']),
    ('Cancelling halfway',
     ['book an appointment', 'id', 'cancel']),
    ('A follow up',
     ['how much is clearance', 'and transcript?']),
    ('Unexpected input, then a person',
     ['', 'asdfgh', 'qwerty', 'human']),
    ('Local phrasing',
     ['ibizamini ni ryari', 'clearance ni angahe', 'murakoze']),
    ('A booking with the service already given',
     ['book a transcript appointment']),
]

number = 1
for title, turns in TESTS:
    print(f"Test {number}: {title}")
    run_conversation(turns)
    number = number + 1

# STEP 9. Refine: test 10 failed. The booking handler ignored the service in the request.
# The change: after starting the flow, use every answer the request already contains.
def handle_book(text):
    """Refined after test 10: answers already in the request are used."""
    if flow["step"] is not None:
        return "Your appointment booking is already under way."
    reply = start_flow()
    while flow["step"] != "confirm":
        pattern = FLOW_STEPS[flow["step"]][2]      # the current question's pattern
        if not re.search(pattern, text):
            break                                  # the request does not answer it: ask
        reply = continue_flow(text)                # it does: answer it, and go on
    return reply

HANDLERS["book"] = handle_book       # the router keeps the old function until this line

title, turns = TESTS[9]
print("Test 10 again:", title)
run_conversation(turns)

# STEP 10. YOUR DOMAIN: messages written the way your users really write:
# Kinyarwanda, mixed, SMS spelling. Then see which ones end up in UNMATCHED.
PHRASING = [
    'ibizamini ni ryari',
    'muraho, clearance ni angahe',
    'amanota yanjye',
    'hw much 4 transcript',
    'book registry 2moro',
    'cancle',
]

def not_understood(messages):
    """Run each message from a clean start. Return the ones that ended up in UNMATCHED."""
    missed = []
    for message in messages:
        reset_flow()
        reset_context()
        before = len(UNMATCHED)
        reply_to(message)
        if len(UNMATCHED) > before:
            missed.append(message)
    reset_flow()
    reset_context()
    return missed

missed = not_understood(PHRASING)
print(len(missed), "of", len(PHRASING), "not understood:", missed)

ready
ULK Exam & Registry Desk: a chatbot for undergraduate students at the Independent University of Kigali (ULK).
It can help with exam timetables, clearance fees, transcript collection and booking a registry appointment.
5 intents in the rule table
[('timetable', 'when are exams'), ('fees', 'how much is clearance')]
5 intents routed
3 questions, then a confirmation
Bot: Which service do you need? Clearance, transcript, resit, or ID?
You: transcript please
Bot: Which day? Today or tomorrow?
You: tomorrow
Bot: Morning or afternoon?
You: morning
Bot: So: transcript, tomorrow, morning. Shall I book it? Yes or no.
You: yes
Bot: Appointment booked: transcript, tomorrow, morning. Thank you.
7 rules, 7 intents routed
validation, error handling and the fallback are ready
You: muraho
Bot: Muraho! I am the ULK Exam & Registry Desk assistant, a computer program. I can help with exam timetables, clearance fees, transcript collection and booking a registry appointment.
You: how much is clearance


---

## Step 2: Your classifier

**YOUR DOMAIN.** Upload `intent_classifier.joblib` from Part 1 to the Files panel, and set `THRESHOLD` to the value chosen in Part 1. The first line of the output says which classifier is in use.

In [7]:
# STEP 2. YOUR DOMAIN: the classifier from Part 1, and the threshold you chose there.
import os
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

THRESHOLD = 0.25      # YOUR DOMAIN: Part 1, Step 6

if os.path.exists("intent_classifier.joblib"):
    classifier = joblib.load("intent_classifier.joblib")
    print("Loaded intent_classifier.joblib")
else:
    print("intent_classifier.joblib NOT FOUND: the worked example is used. It may not be submitted.")
    EXAMPLE = [('what time does the first bus leave', 'hours'), ('bus to huye leaves when', 'hours'), ('saa ngahe bus igenda?', 'hours'), ('wen is the last bus', 'hours'), ('are you open on sunday', 'hours'), ('timetable pls', 'hours'), ('when does the office open', 'hours'), ('last bus to musanze tonight?', 'hours'), ('what time do you close', 'hours'), ('is there a bus at 6am', 'hours'), ('bus igenda saa ngahe ejo', 'hours'), ('r u open today', 'hours'), ('how early do buses start', 'hours'), ('departure times please', 'hours'), ('when is the next bus', 'hours'), ('how much to musanze', 'fares'), ('ticket price for rubavu?', 'fares'), ('itike ya huye ni angahe', 'fares'), ('hw mch 2 nyagatare', 'fares'), ('is there a student discount', 'fares'), ('what does a ticket cost', 'fares'), ('price to huye pls', 'fares'), ('how much is a return ticket', 'fares'), ('fare for a child', 'fares'), ('itike ni angahe', 'fares'), ('is the fare cheaper at night', 'fares'), ('cost of a ticket to rubavu', 'fares'), ('how many francs to musanze', 'fares'), ('fares list', 'fares'), ('do kids pay full price', 'fares'), ('can i bring two bags', 'luggage'), ('how heavy can my suitcase be', 'luggage'), ('is the bag fee extra', 'luggage'), ('can i carry a bicycle', 'luggage'), ('imizigo yemewe ni ingahe?', 'luggage'), ('i have a big box, is that ok', 'luggage'), ('luggage limit?', 'luggage'), ('can i take a sack of potatoes', 'luggage'), ('is there space for a suitcase', 'luggage'), ('do you charge for bags', 'luggage'), ('can i bring my guitar', 'luggage'), ('how many bags allowed', 'luggage'), ('big luggage ok?', 'luggage'), ('nfite imizigo myinshi', 'luggage'), ('can i send a parcel on the bus', 'luggage'), ('book a seat to huye', 'book'), ('i want to travel to musanze tomorrow', 'book'), ('reserve 2 seats for rubavu', 'book'), ('ndashaka itike ya huye', 'book'), ('get me on the morning bus', 'book'), ('can i book for friday', 'book'), ('one ticket to nyagatare please', 'book'), ('i need a seat on the 8am bus', 'book'), ('book me to rubavu', 'book'), ('reserve a ticket for sunday', 'book'), ('ndashaka kujya musanze', 'book'), ('two tickets to huye today', 'book'), ('can i get a seat tonight', 'book'), ('i want to go to rubavu on saturday', 'book'), ('booking for 3 people', 'book'), ('cancel my booking', 'cancel'), ('i cant travel anymore', 'cancel'), ('call off my seat', 'cancel'), ('ndashaka guhagarika itike', 'cancel'), ('i want my money back', 'cancel'), ('cancle the ticket', 'cancel'), ('cancel my seat to huye', 'cancel'), ('refund please', 'cancel'), ('i will not travel', 'cancel'), ('drop my booking', 'cancel'), ('cancel the trip', 'cancel'), ('hagarika itike yanjye', 'cancel'), ('i changed my plans, cancel', 'cancel'), ('stop my reservation', 'cancel'), ('cancl my ticket', 'cancel'), ('i lost my phone on the bus', 'out_of_scope'), ('are you hiring drivers', 'out_of_scope'), ('the driver was rude', 'out_of_scope'), ('can i pay with momo', 'out_of_scope'), ('who won the match', 'out_of_scope'), ('asdfgh', 'out_of_scope'), ('where is the bus park', 'out_of_scope'), ('the bus was dirty', 'out_of_scope'), ('i want to speak to the manager', 'out_of_scope'), ('tell me a joke', 'out_of_scope'), ('is there wifi on the bus', 'out_of_scope'), ('my seat was broken', 'out_of_scope'), ('ok', 'out_of_scope'), ('what is your name', 'out_of_scope'), ('i left my bag on the bus', 'out_of_scope')]
    classifier = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)),
                               LogisticRegression(max_iter=1000))
    classifier.fit([t for t, _ in EXAMPLE], [i for _, i in EXAMPLE])


def classify(text):
    """The intent with the highest score, and that score."""
    scores = classifier.predict_proba([text])[0]
    best = scores.argmax()
    return str(classifier.classes_[best]), float(scores[best])


print("intents:", list(classifier.classes_))

intent_classifier.joblib NOT FOUND: the worked example is used. It may not be submitted.
intents: [np.str_('book'), np.str_('cancel'), np.str_('fares'), np.str_('hours'), np.str_('luggage'), np.str_('out_of_scope')]


In [8]:
# STEP 3. PROVIDED: the classifier inside the chatbot (Week 9's hybrid, for any chatbot).
if "route_rules" not in globals():       # run again safely: keep the original rules
    route_rules = route

def route(message):
    """Rules keep the intents the classifier never learned; the classifier picks the rest."""
    found = []
    for request in split_requests(message):
        tag = None
        rule_only = [t for t, _ in route_rules(request) if t not in classifier.classes_]
        if rule_only:
            tag = rule_only[0]
        else:
            intent, score = classify(request)
            if score >= THRESHOLD and intent != "out_of_scope":
                tag = intent
        if tag in HANDLERS and tag not in [t for t, _ in found]:
            found.append((tag, request))
    return found               # empty: the chatbot's own fallback answers

TRY = [
    "hw mch 4 lost id replacement",
    "saa ngahe ibizamini bitangira",
    "ndashaka gufata gahunda ku ishuri",
    "where is the campus cafeteria",
]
for m in TRY:
    print(f"{m:36} rules: {[t for t, _ in route_rules(m)]}  classifier: {route(m)}")



hw mch 4 lost id replacement         rules: []  classifier: []
saa ngahe ibizamini bitangira        rules: ['timetable']  classifier: [('timetable', 'saa ngahe ibizamini bitangira')]
ndashaka gufata gahunda ku ishuri    rules: []  classifier: [('book', 'ndashaka gufata gahunda ku ishuri')]
where is the campus cafeteria        rules: []  classifier: []


---

## Step 4: One memory per chat, and the handlers

**PROVIDED.** Each Telegram chat keeps its own conversation. The handlers are the Week 10 ones; `START` is built from `BOT_NAME` and `TOPICS`.

In [9]:
# STEP 4. PROVIDED: one memory per chat, and the Telegram handlers (Week 10).
import copy

SESSIONS = {}          # chat id: that customer's own flow and context


def reply_for(chat_id, text):
    """One turn for ONE customer: load their memory, answer, save it back."""
    if chat_id not in SESSIONS:
        reset_flow()                     # a new customer starts clean
        reset_context()
    else:
        saved_flow, saved_context = SESSIONS[chat_id]
        flow.clear()
        flow.update(copy.deepcopy(saved_flow))
        context.clear()
        context.update(copy.deepcopy(saved_context))
    answer = reply_to(text)
    SESSIONS[chat_id] = (copy.deepcopy(flow), copy.deepcopy(context))
    return answer


START = (f"Hello! I am the {BOT_NAME} assistant: a computer program. "
         f"I can help with {TOPICS}.")


async def on_start(update, ctx):
    """/start and /help: say what the bot is, and what it can do."""
    await update.message.reply_text(START)


async def on_text(update, ctx):
    """Every text message: the chatbot answers, with this chat's memory."""
    chat_id = update.effective_chat.id
    answer = reply_for(chat_id, update.message.text)
    await update.message.reply_text(answer)


async def on_error(update, ctx):
    """Anything that goes wrong is logged, and the bot keeps running."""
    ERROR_LOG.append(repr(ctx.error))
    print("Error:", ctx.error)

The handlers, tested with fake updates before Telegram.

In [11]:
# Test the handlers WITHOUT Telegram: fake updates with the same shape.
from types import SimpleNamespace


class FakeMessage:
    def __init__(self, text):
        self.text = text
        self.sent = []

    async def reply_text(self, answer):
        self.sent.append(answer)


async def send(chat_id, text):
    message = FakeMessage(text)
    update = SimpleNamespace(message=message,
                             effective_chat=SimpleNamespace(id=chat_id))
    if text.startswith("/"):
        await on_start(update, None)
    else:
        await on_text(update, None)
    return message.sent[0]


SESSIONS.clear()
for chat_id, text in [(3333, "/start"), (3333, "hw mch 2 rubavu"),
                      (4444, "book a seat"), (3333, "who won the match")]:
    print(text, "->", await send(chat_id, text))

/start -> Hello! I am the ULK Exam & Registry Desk assistant: a computer program. I can help with exam timetables, clearance fees, transcript collection and booking a registry appointment.
hw mch 2 rubavu -> Sorry, I did not understand. I can help with exam timetables, clearance fees, transcript collection and booking a registry appointment.
book a seat -> Which service do you need? Clearance, transcript, resit, or ID?
who won the match -> Sorry, I still did not understand. Type 'human' for a member of staff, or ask about exam timetables, clearance fees, transcript collection and booking a registry appointment.


In [2]:
%pip install -q python-telegram-bot

---

## Step 5: On Telegram

**PROVIDED.** Make the bot with @BotFather (`/newbot`), put the token in Colab's secrets as `TELEGRAM_TOKEN` with Notebook access on, then run these cells. They have no saved output: they need the token.

**Streamlit instead of Telegram?** Skip Steps 5 and 6: follow `How to deploy.md` in the Streamlit starter folder.

The token, from the secrets. It is never printed.

In [13]:
import os

TELEGRAM_TOKEN = os.environ.get("TELEGRAM_TOKEN")
try:
    from google.colab import userdata
    TELEGRAM_TOKEN = userdata.get("TELEGRAM_TOKEN")
except Exception as error:      # not in Colab, or the secret is missing
    print("Colab secret not read:", type(error).__name__)

# Never print the token itself: it controls the bot.
print("Token loaded." if TELEGRAM_TOKEN else "No token yet: see the steps above.")

Token loaded.


Start the bot, then talk to it from a phone. Take the screenshots now.

In [14]:
from telegram.ext import Application, CommandHandler, MessageHandler, filters

app = Application.builder().token(TELEGRAM_TOKEN).build()
app.add_handler(CommandHandler(["start", "help"], on_start))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, on_text))
app.add_error_handler(on_error)

# Colab is already running an event loop, so app.run_polling() cannot be
# used in a notebook. These three lines start the bot inside that loop.
await app.initialize()
await app.start()
await app.updater.start_polling()
print("The bot is running. Message it on Telegram; run the next cell to stop it.")

The bot is running. Message it on Telegram; run the next cell to stop it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Stop it before starting it again.

In [20]:
await app.updater.stop()
await app.stop()
await app.shutdown()
print("The bot has stopped.")

The bot has stopped.


---

## Step 6: The README, then publish and submit

In the `telegram` folder of your Portfolio Project 2 repository, add a `README.md` with a section
**On Telegram**:

| | |
|---|---|
| The bot | its Telegram username |
| Three classified intents | a screenshot of each answer, and the phrasing the rules alone missed |
| The fallback | a screenshot of a message out of scope, and the reply |
| Two chats | a screenshot of two chats (two phones, or a phone and web.telegram.org) at the same time |
| Threshold | the value from Part 1, and what it did on Telegram |
| One thing that failed | a message the bot got wrong on Telegram, and why |

1. Upload this notebook (File, then Download, then .ipynb), `intent_classifier.joblib` and the screenshots to
   the `telegram` folder. Search the notebook for the token before uploading: it must not be there.
2. Open the folder in a private browsing window. Paste its URL in Google Classroom and click
   **Turn in**, by **Monday 16 November 2026, 23:59**.